# Shreve Week 05 — Itô's Lemma and Applications

**Week 5** — Itô's Lemma and Applications

This notebook teaches **itô's lemma and applications** in the style of our graduate probability notebook: definitions from **Shreve**, intuition and examples from **Baxter & Rennie**, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve | Baxter & Rennie |
|------|-------|--------|-----------------|
| 1 | **Itô's lemma (1D)** | Ch. 4.4 | Ch. 3.3 |
| 2 | **$dW_t^2 = 2W_t dW_t + dt$** | Ch. 4.4 | Ch. 3.3 |
| 3 | **Geometric Brownian motion** | Ch. 4.4 | Ch. 3.3 |
| 4 | **Log-price dynamics** | Ch. 4.4 | Ch. 3.3 |
| — | **Problem set** | Ch. 4 | App. 3 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. **Shreve** (*Stochastic Calculus for Finance II*) — rigorous measure-theoretic treatment; see chapter pointers in each section.
5. **Baxter & Rennie** (*Financial Calculus*) — market intuition, replication, and worked examples; see spotlight sections.

**References:**
- **Shreve** Vol II — Ch. 4 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Baxter & Rennie**, *Financial Calculus* — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Itô's Lemma

For smooth $f(t,x)$ and $X_t$ satisfying $dX_t = \mu_t\, dt + \sigma_t\, dW_t$:

$$df(t, X_t) = \frac{\partial f}{\partial t} dt + \frac{\partial f}{\partial x} dX_t + \frac{1}{2}\frac{\partial^2 f}{\partial x^2} (dX_t)^2$$

With $(dW)^2 = dt$, $(dt)^2 = 0$, $dW\, dt = 0$.

**Shreve Ch. 4.4:** Itô's lemma — the chain rule of stochastic calculus.


In [ ]:
# Verify Itô for f(W) = W²: df = 2W dW + dt
rng = np.random.default_rng(40)
T, n = 1.0, 2000
dt = T / n
dW = rng.normal(0, np.sqrt(dt), size=n)
W = np.concatenate([[0], np.cumsum(dW)])

f = W**2
df_actual = f[1:] - f[:-1]
df_ito = 2 * W[:-1] * dW + dt

print(f"Mean df (actual): {df_actual.mean():.6f}")
print(f"Mean df (Itô):    {df_ito.mean():.6f} (theory dt={dt})")
print(f"Corr(actual, Itô): {np.corrcoef(df_actual, df_ito)[0,1]:.4f}")


---
# Part 2 — Application: $W_t^2 - t$ Martingale

Itô on $f(W) = W^2$: $d(W^2) = 2W\, dW + dt$, so $d(W^2 - t) = 2W\, dW$ — pure martingale part.

**Shreve Ch. 4.4:** building martingales via Itô's lemma.


In [ ]:
rng = np.random.default_rng(41)
T, n_steps = 1.0, 1000
dt = T / n_steps
n_paths = 50
fig, ax = plt.subplots()
for _ in range(n_paths):
    dW = rng.normal(0, np.sqrt(dt), size=n_steps)
    W = np.concatenate([[0], np.cumsum(dW)])
    t = np.linspace(0, T, n_steps + 1)
    ax.plot(t, W**2 - t, alpha=0.7)
ax.axhline(0, color="k", lw=1)
ax.set_title("W_t² - t (martingale paths)")
plt.show()


---
# Part 3 — Geometric Brownian Motion

Stock model: $dS_t = \mu S_t\, dt + \sigma S_t\, dW_t$.

Itô on $\log S$: $d\log S_t = (\mu - \sigma^2/2) dt + \sigma\, dW_t$.

Solution: $S_t = S_0 \exp\big((\mu - \sigma^2/2)t + \sigma W_t\big)$.

**Shreve Ch. 4.4:** GBM as application of Itô's lemma.


In [ ]:
def simulate_gbm(S0, mu, sigma, T, n_steps, seed=42):
    rng = np.random.default_rng(seed)
    dt = T / n_steps
    dW = rng.normal(0, np.sqrt(dt), size=n_steps)
    log_S = np.log(S0) + (mu - sigma**2/2)*dt + sigma*dW
    S = np.exp(np.concatenate([[np.log(S0)], np.log(S0) + np.cumsum(
        (mu - sigma**2/2)*dt + sigma*dW)]))
    return S

S0, mu, sigma, T = 100, 0.08, 0.2, 1.0
paths = [simulate_gbm(S0, mu, sigma, T, 252, seed=s) for s in range(20)]
fig, ax = plt.subplots()
for p in paths:
    ax.plot(p, alpha=0.7)
ax.set_title(f"GBM paths: μ={mu}, σ={sigma}")
plt.show()
print(f"E[S_T] theory = {S0*np.exp(mu*T):.2f}")


---
# Part 4 — Exponential Martingale

$M_t = \exp(\sigma W_t - \frac{1}{2}\sigma^2 t)$ satisfies $dM_t = \sigma M_t\, dW_t$ — a martingale.

**Shreve Ch. 4.4:** exponential martingale (Radon-Nikodym preview).


In [ ]:
rng = np.random.default_rng(43)
sigma, T, n_steps = 0.3, 1.0, 1000
dt = T / n_steps
n_sims = 10_000
M_T = []
for _ in range(n_sims):
    dW = rng.normal(0, np.sqrt(dt), size=n_steps)
    W_T = np.sum(dW)
    M_T.append(np.exp(sigma * W_T - 0.5 * sigma**2 * T))
print(f"E[M_T] sim = {np.mean(M_T):.4f} (theory 1.0 martingale)")


**Your turn:** Apply Itô to $f(S) = S^2$ when $dS = \mu S\, dt + \sigma S\, dW$.


---
## Baxter & Rennie spotlight — Itô calculus (Ch. 3.3)

B&R present **Itô's lemma** as the chain rule for functions of stochastic processes, emphasizing the extra $\frac{1}{2} f'' (dX)^2$ term from nonzero quadratic variation.

They derive **geometric Brownian motion** by applying Itô to $\log S_t$.

**Read:** B&R Ch. 3.3; compare notation with Shreve Ch. 4.4.


**B&R drill:** For $f(x) = e^x$ and $dX_t = \mu\, dt + \sigma\, dW_t$, Itô gives $df = e^X(\mu + \frac{1}{2}\sigma^2)\, dt + \sigma e^X\, dW$.


In [ ]:
# B&R Ch. 3.3 — Itô on exp(X)
rng = np.random.default_rng(40)
mu, sigma, T, n = 0.05, 0.3, 1.0, 5000
dt = T / n
dW = rng.normal(0, np.sqrt(dt), size=n)
X = np.cumsum(mu*dt + sigma*dW)
f = np.exp(X)
df = f[1:] - f[:-1]
df_ito = f[:-1] * (mu + 0.5*sigma**2)*dt + f[:-1]*sigma*dW
print(f"Corr(actual df, Itô): {np.corrcoef(df, df_ito)[0,1]:.4f}")


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Apply Itô's lemma to $f(x) = e^x$ with $dX = \mu dt + \sigma dW$.

2. Derive $d(S_t^2)$ for GBM.

3. Show $Y_t = e^{\sigma W_t - \frac{1}{2}\sigma^2 t}$ is a martingale.

4. For $f(t,W_t) = t W_t$, compute $df$ using Itô's lemma.


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** $df = e^X(\mu + \frac{1}{2}\sigma^2) dt + \sigma e^X dW$.

**2.** $d(S^2) = 2S dS + (dS)^2 = (2\mu + \sigma^2)S^2 dt + 2\sigma S^2 dW$.

**3.** Itô on log gives $d\log Y = \sigma dW$, so $Y$ is stochastic exponential of $\sigma W$.

**4.** $df = W_t dt + t dW_t$.

</details>


---
## Further reading

### Shreve
- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 4 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

### Baxter & Rennie
- **Baxter & Rennie**, Ch. 3.3 — [PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Baxter_Rennie_Financial_Calculus.pdf): Itô's lemma, GBM derivation.
- B&R worked examples use the same $(dW)^2 = dt$ rules as Shreve.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
